# Alternative Document Extraction Methods for I-9 Forms

This notebook demonstrates 4 different extraction methods for documents that are difficult to parse with standard text extraction but can be filled neatly.

## Methods:
1. **OCR-Based Extraction** - Visual text recognition
2. **Template-Based Extraction** - Coordinate-based field extraction
3. **Pattern-Matching Extraction** - Advanced regex patterns
4. **Hybrid Approach** - Combining multiple methods

In [ ]:
# Install dependencies
!pip install opencv-python pytesseract pillow numpy pandas matplotlib PyMuPDF

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytesseract
import fitz  # PyMuPDF
import re
import json
from typing import Dict, List, Tuple
from dataclasses import dataclass
from enum import Enum

print("✅ Dependencies loaded!")

In [ ]:
# Data structures
class FieldType(Enum):
    TEXT = "text"
    DATE = "date"
    PHONE = "phone"
    EMAIL = "email"
    SSN = "ssn"
    ADDRESS = "address"

@dataclass
class ExtractedField:
    name: str
    value: str
    confidence: float
    field_type: FieldType
    method: str

## Method 1: OCR-Based Extraction

In [ ]:
class OCRExtractor:
    def __init__(self):
        self.patterns = {
            'phone': r'(\(?[0-9]{3}\)?[-\.\s]?[0-9]{3}[-\.\s]?[0-9]{4})',
            'email': r'([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})',
            'ssn': r'([0-9]{3}[-\s][0-9]{2}[-\s][0-9]{4})',
            'date': r'([0-9]{1,2}[/\-][0-9]{1,2}[/\-][0-9]{2,4})'
        }
    
    def extract_from_pdf(self, pdf_path: str) -> Dict[str, ExtractedField]:
        print(f"🔍 OCR Extraction: {os.path.basename(pdf_path)}")
        
        # Convert PDF to image
        doc = fitz.open(pdf_path)
        page = doc[0]
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img_data = pix.tobytes("png")
        
        # Convert to OpenCV format
        nparr = np.frombuffer(img_data, np.uint8)
        image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        # Preprocess for OCR
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
        
        # Extract text
        text = pytesseract.image_to_string(thresh)
        
        # Apply patterns
        extracted = {}
        for field_name, pattern in self.patterns.items():
            matches = re.findall(pattern, text)
            if matches:
                extracted[field_name] = ExtractedField(
                    name=field_name,
                    value=matches[0],
                    confidence=0.85,
                    field_type=FieldType(field_name.upper() if field_name.upper() in [e.name for e in FieldType] else "TEXT"),
                    method="OCR"
                )
        
        doc.close()
        print(f"   ✅ Extracted {len(extracted)} fields")
        return extracted

# Test OCR Extractor
ocr_extractor = OCRExtractor()
print("OCR Extractor ready!")

## Method 2: Template-Based Extraction

In [ ]:
class TemplateExtractor:
    def __init__(self):
        # I-9 form field coordinates (percentage of page)
        self.i9_template = {
            'last_name': {'x': 0.15, 'y': 0.25, 'width': 0.25, 'height': 0.03},
            'first_name': {'x': 0.45, 'y': 0.25, 'width': 0.25, 'height': 0.03},
            'address': {'x': 0.15, 'y': 0.30, 'width': 0.7, 'height': 0.03},
            'phone': {'x': 0.15, 'y': 0.45, 'width': 0.2, 'height': 0.03},
            'email': {'x': 0.65, 'y': 0.40, 'width': 0.2, 'height': 0.03}
        }
    
    def extract_from_pdf(self, pdf_path: str) -> Dict[str, ExtractedField]:
        print(f"📋 Template Extraction: {os.path.basename(pdf_path)}")
        
        # Convert PDF to image
        doc = fitz.open(pdf_path)
        page = doc[0]
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img_data = pix.tobytes("png")
        
        nparr = np.frombuffer(img_data, np.uint8)
        image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        extracted = {}
        page_height, page_width = image.shape[:2]
        
        for field_name, coords in self.i9_template.items():
            # Convert percentage to pixels
            x = int(coords['x'] * page_width)
            y = int(coords['y'] * page_height)
            w = int(coords['width'] * page_width)
            h = int(coords['height'] * page_height)
            
            # Extract region
            roi = image[y:y+h, x:x+w]
            
            # OCR on region
            gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            text = pytesseract.image_to_string(gray, config='--psm 8').strip()
            
            if text and len(text) > 1:
                extracted[field_name] = ExtractedField(
                    name=field_name,
                    value=text,
                    confidence=0.90,
                    field_type=FieldType.TEXT,
                    method="Template"
                )
        
        doc.close()
        print(f"   ✅ Extracted {len(extracted)} fields")
        return extracted

# Test Template Extractor
template_extractor = TemplateExtractor()
print("Template Extractor ready!")

## Method 3: Pattern-Matching Extraction

In [ ]:
class PatternExtractor:
    def __init__(self):
        self.advanced_patterns = {
            'name': [
                r'(?:name|employee)[:\s]*([A-Za-z\s]{2,30})',
                r'([A-Z][a-z]+\s+[A-Z][a-z]+)'
            ],
            'phone': [
                r'(?:phone|tel)[:\s]*(\(?[0-9]{3}\)?[-\.\s]?[0-9]{3}[-\.\s]?[0-9]{4})',
                r'(\(?[0-9]{3}\)?[-\.\s]?[0-9]{3}[-\.\s]?[0-9]{4})'
            ],
            'email': [
                r'(?:email|e-mail)[:\s]*([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})',
                r'([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})'
            ],
            'address': [
                r'(?:address|street)[:\s]*([0-9]+\s+[A-Za-z\s,]{5,50})',
                r'([0-9]+\s+[A-Za-z\s,]{10,50})'
            ]
        }
    
    def extract_from_pdf(self, pdf_path: str) -> Dict[str, ExtractedField]:
        print(f"🔍 Pattern Extraction: {os.path.basename(pdf_path)}")
        
        # Extract text from PDF
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        
        extracted = {}
        
        for field_name, patterns in self.advanced_patterns.items():
            best_match = None
            best_confidence = 0.0
            
            for pattern in patterns:
                matches = re.finditer(pattern, text, re.IGNORECASE)
                for match in matches:
                    value = match.group(1) if match.groups() else match.group(0)
                    value = value.strip()
                    
                    if len(value) > 1:
                        confidence = self._calculate_confidence(field_name, value)
                        if confidence > best_confidence:
                            best_match = value
                            best_confidence = confidence
            
            if best_match and best_confidence > 0.6:
                extracted[field_name] = ExtractedField(
                    name=field_name,
                    value=best_match,
                    confidence=best_confidence,
                    field_type=self._get_field_type(field_name),
                    method="Pattern"
                )
        
        doc.close()
        print(f"   ✅ Extracted {len(extracted)} fields")
        return extracted
    
    def _calculate_confidence(self, field_name: str, value: str) -> float:
        base_confidence = 0.7
        
        if field_name == 'phone' and re.match(r'\(?[0-9]{3}\)?[-\.\s]?[0-9]{3}[-\.\s]?[0-9]{4}', value):
            return 0.95
        elif field_name == 'email' and '@' in value and '.' in value:
            return 0.95
        
        return base_confidence
    
    def _get_field_type(self, field_name: str) -> FieldType:
        mapping = {
            'phone': FieldType.PHONE,
            'email': FieldType.EMAIL,
            'address': FieldType.ADDRESS
        }
        return mapping.get(field_name, FieldType.TEXT)

# Test Pattern Extractor
pattern_extractor = PatternExtractor()
print("Pattern Extractor ready!")

## Method 4: Hybrid Approach

In [ ]:
class HybridExtractor:
    def __init__(self):
        self.ocr_extractor = OCRExtractor()
        self.template_extractor = TemplateExtractor()
        self.pattern_extractor = PatternExtractor()
    
    def extract_from_pdf(self, pdf_path: str) -> Dict[str, ExtractedField]:
        print(f"🚀 Hybrid Extraction: {os.path.basename(pdf_path)}")
        
        # Run all extractors
        ocr_results = self.ocr_extractor.extract_from_pdf(pdf_path)
        template_results = self.template_extractor.extract_from_pdf(pdf_path)
        pattern_results = self.pattern_extractor.extract_from_pdf(pdf_path)
        
        # Combine results (highest confidence wins)
        combined = {}
        
        for results in [ocr_results, template_results, pattern_results]:
            for field_name, field in results.items():
                if field_name not in combined or field.confidence > combined[field_name].confidence:
                    combined[field_name] = field
        
        print(f"   ✅ Combined {len(combined)} fields from all methods")
        return combined

# Test Hybrid Extractor
hybrid_extractor = HybridExtractor()
print("Hybrid Extractor ready!")

## Demo: Extract from I-9 Form

In [ ]:
# Test with I-9 form (adjust path as needed)
pdf_path = "../Forms/i-9-1.pdf"  # or "../Documents/i-9-1.pdf"

if os.path.exists(pdf_path):
    print("Testing all extraction methods on I-9 form...\n")
    
    # Test each method
    methods = {
        "OCR": ocr_extractor,
        "Template": template_extractor,
        "Pattern": pattern_extractor,
        "Hybrid": hybrid_extractor
    }
    
    results = {}
    
    for method_name, extractor in methods.items():
        print(f"\n=== {method_name} Method ===")
        extracted = extractor.extract_from_pdf(pdf_path)
        results[method_name] = extracted
        
        for field_name, field in extracted.items():
            print(f"  {field_name}: {field.value} (confidence: {field.confidence:.2f})")
    
    # Create comparison DataFrame
    comparison_data = []
    all_fields = set()
    
    for method_results in results.values():
        all_fields.update(method_results.keys())
    
    for field in all_fields:
        row = {'Field': field}
        for method_name, method_results in results.items():
            if field in method_results:
                row[method_name] = f"{method_results[field].value} ({method_results[field].confidence:.2f})"
            else:
                row[method_name] = "Not found"
        comparison_data.append(row)
    
    df = pd.DataFrame(comparison_data)
    print("\n=== Comparison of All Methods ===")
    print(df.to_string(index=False))
    
else:
    print(f"PDF file not found: {pdf_path}")
    print("Please place an I-9 form in the Forms or Documents folder")

## Export Results

In [ ]:
def export_results(results: Dict[str, Dict[str, ExtractedField]], output_file: str):
    """Export extraction results to JSON"""
    export_data = {}
    
    for method_name, method_results in results.items():
        export_data[method_name] = {}
        for field_name, field in method_results.items():
            export_data[method_name][field_name] = {
                'value': field.value,
                'confidence': field.confidence,
                'type': field.field_type.value,
                'method': field.method
            }
    
    with open(output_file, 'w') as f:
        json.dump(export_data, f, indent=2)
    
    print(f"Results exported to: {output_file}")

# Export results if we have them
if 'results' in locals():
    export_results(results, "extraction_results.json")

## Summary

This notebook demonstrates 4 alternative extraction methods for difficult-to-parse documents:

1. **OCR-Based**: Uses visual text recognition with pattern matching
2. **Template-Based**: Uses predefined coordinates for specific form types
3. **Pattern-Matching**: Uses advanced regex patterns for structured data
4. **Hybrid**: Combines all methods for best results

Each method has its strengths:
- OCR works well for clear, printed text
- Template works best for consistent form layouts
- Pattern matching is flexible for various document types
- Hybrid approach provides the most comprehensive extraction

The hybrid approach typically provides the best results by leveraging the strengths of all methods.